# TrustedSQL Intent-GNN Training
Notebook gọi đúng lifecycle CLI; checkpoint candidate không tự động trở thành runtime checkpoint.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, subprocess, sys
ROOT = Path(os.environ.get('TRUSTEDSQL_PROJECT_ROOT', Path.cwd())).resolve()
if not (ROOT / 'pyproject.toml').exists(): raise RuntimeError('Set TRUSTEDSQL_PROJECT_ROOT')
RUN_ID = f"gnn_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DEVICE = 'cuda'
EPOCHS = 30
ROOT, RUN_ID

In [ ]:
def run(*args):
    cmd = [sys.executable, '-m', 'trustedsql_gnn.cli', '--project-root', str(ROOT), *args]
    completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print(completed.stdout); print(completed.stderr) if completed.stderr else None
    completed.check_returncode()

## Inspect và prepare

In [ ]:
run('inspect')
run('prepare', '--run-id', RUN_ID)

## Train và evaluate candidate

In [ ]:
run('train', '--run-id', RUN_ID, '--device', DEVICE, '--epochs', str(EPOCHS))
candidate = ROOT / 'outputs' / 'training' / RUN_ID / 'checkpoints' / 'candidate.pt'
run('evaluate', '--run-id', RUN_ID, '--checkpoint', str(candidate), '--device', DEVICE)

## Promotion gate
Chỉ promote sau khi kiểm tra training report, test/hard-holdout report và provenance. Chạy CLI `promote` thủ công với `--confirmed-by`.